# 🌊 CoastSeg: Automated Tidal Correction & Portal Export

Notebook ini digunakan setelah Anda melakukan **Extract Shorelines** di GUI CoastSeg. 
Notebook ini akan otomatis:
1. Memprediksi Pasang Surut (FES2022).
2. Mengestimasi Beach Slope per Transek.
3. Melakukan Koreksi Pasang Surut pada garis pantai.
4. Mengekspor data untuk Portal Laravel.

In [1]:
import os
import sys

# 1. PASTIKAN MEMBACA DARI SRC/COASTSEG
project_root = os.path.abspath(os.getcwd())
src_path = os.path.join(project_root, 'src')

if src_path not in sys.path:
    sys.path.insert(0, src_path)

import coastseg
print(f"CoastSeg loaded from: {coastseg.__file__}")

from coastseg import tide_correction
import prepare_portal_data
import generate_tides_for_session

CoastSeg loaded from: /home/kkp/projects/iaams/CoastSeg/src/coastseg/__init__.py


/home/kkp/miniforge3/envs/coastseg/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 2. Pilih Session Anda
Ganti `session_path` di bawah ini dengan folder session hasil extraction Anda.

In [2]:
session_path = "sessions/ID_Jawa-17_datetime04-07-26__11_43_53/"

if not os.path.exists(session_path):
    print("❌ Path tidak ditemukan! Pastikan path benar.")
else:
    print(f"✅ Session terpilih: {session_path}")

✅ Session terpilih: sessions/ID_Jawa-17_datetime04-07-26__11_43_53/


## 3. Otomatisasi Persiapan Data (Tides & Slopes)
Langkah ini akan menghasilkan file `predicted_tides.csv` dan `estimated_beach_slopes.csv` secara otomatis.

In [3]:
print("🚀 Memulai prediksi pasang surut dan estimasi slope...")
generate_tides_for_session.prepare_tides_and_slopes(session_path)
print("✅ Persiapan data selesai.")

🚀 Memulai prediksi pasang surut dan estimasi slope...
Processing session: sessions/ID_Jawa-17_datetime04-07-26__11_43_53/
Loading data...
Transect IDs available in timeseries: ['2' '7' '83' '82' '9' '81' '11' '12' '16' '18']... (Total: 63)
Predicting tides for 63 transects...


  Predicting tides for 63 transects: 100%|██████████████████████████████████████| 63/63 [06:59<00:00,  6.66s/it]


Saved predicted tides to sessions/ID_Jawa-17_datetime04-07-26__11_43_53/predicted_tides.csv (Long Format)
Estimating beach slopes...


Estimating Slopes:  73%|████████████████████████████████████████▉               | 46/63 [00:02<00:00, 17.63it/s]

Estimating Slopes:  95%|█████████████████████████████████████████████████████▎  | 60/63 [00:03<00:00, 22.69it/s]

Estimating Slopes: 100%|████████████████████████████████████████████████████████| 63/63 [00:03<00:00, 16.30it/s]

Saved estimated slopes to sessions/ID_Jawa-17_datetime04-07-26__11_43_53/estimated_beach_slopes.csv
   transect_id  slope
0            2  0.200
1            7  0.200
2           83  0.200
3           82  0.200
4            9  0.200
5           81  0.200
6           11  0.200
7           12  0.200
8           16  0.200
9           18  0.200
10          23  0.200
11          66  0.200
12          25  0.200
13          56  0.200
14          52  0.112
15           5  0.200
16          26  0.200
17          48  0.200
18          28  0.200
19          47  0.200
20          46  0.052
21          45  0.200
22          29  0.200
23          43  0.200
24          30  0.200
25          42  0.200
26          31  0.200
27          37  0.200
28          32  0.200
29          36  0.200
30          34  0.200
31          27  0.200
32           3  0.200
33          35  0.200
34          72  0.200
35          84  0.190
36          51  0.200
37          88  0.200
38          85  0.200
39          73  0.20

## 4. Jalankan Tidal Correction
Langkah ini akan mengoreksi posisi garis pantai berdasarkan pasang surut dan kemiringan pantai yang telah dihitung.

In [4]:
import geopandas as gpd

# Ambil ROI IDs secara otomatis dari config_gdf
config_gdf = gpd.read_file(os.path.join(session_path, 'config_gdf.geojson'))
roi_ids = config_gdf[config_gdf['type'] == 'roi']['id'].unique().tolist()

# Sesuaikan session_name (biasanya nama folder terakhir)
session_name = os.path.basename(session_path.rstrip('/'))

tides_file = os.path.join(session_path, "predicted_tides.csv")
slopes_file = os.path.join(session_path, "estimated_beach_slopes.csv")

print(f"🛠️ Menjalankan Tidal Correction untuk ROI: {roi_ids}...")
tide_correction.correct_all_tides(
    roi_ids=roi_ids,
    session_name=session_name,
    beach_slope=slopes_file,
    tides_file=tides_file,
    reference_elevation=0
)
print("✅ Tidal Correction selesai.")

🛠️ Menjalankan Tidal Correction untuk ROI: ['Jawa-17']...
Tide model fes2022 found at: '/home/kkp/projects/iaams/CoastSeg/tide_model' and is valid.


Correcting Tides for 1 ROIs:   0%|                                                        | 0/1 [00:00<?, ?it/…

  0%|                                                                                     | 0/6 [00:00<?, ?it/…

/home/kkp/projects/iaams/CoastSeg/src/coastseg/tide_correction.py:397: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  timeseries[column_name].fillna(median_slope, inplace=True)


✅ Tidal Correction selesai.


## 5. Ekspor Data ke Portal Laravel
Menghasilkan file GeoJSON dan JSON di folder `portal_output`.

In [5]:
print("📦 Mengekspor data portal...")
prepare_portal_data.prepare_portal_data(session_path)
print(f"🏁 Selesai! Data portal siap di: {session_path}/portal_output")

📦 Mengekspor data portal...
Preparing portal data for session: sessions/ID_Jawa-17_datetime04-07-26__11_43_53/
Using timeseries file: sessions/ID_Jawa-17_datetime04-07-26__11_43_53/tidally_corrected_transect_time_series_merged.csv
Processing 88 transects...
Portal data successfully exported to: sessions/ID_Jawa-17_datetime04-07-26__11_43_53/portal_output
🏁 Selesai! Data portal siap di: sessions/ID_Jawa-17_datetime04-07-26__11_43_53//portal_output


## 6. Ekspor Global (Gabungan Semua Session)
Menghasilkan 2 kategori: `raw_data` dan `tidally_corrected` yang berisi gabungan dari semua session.

In [6]:
import importlib
importlib.reload(prepare_portal_data)

print("🌍 Memulai Ekspor Global...")
output_base = "global_portal_output"
prepare_portal_data.prepare_global_portal_data("sessions", output_base)
print(f"🏁 Ekspor Global selesai! Hasil ada di folder: {output_base}")

🌍 Memulai Ekspor Global...

--- Aggregating for category: raw_data ---
Processing session: ID_Jawa-18_datetime04-15-26__10_14_08
Processing session: ID_Jawa-17_datetime04-07-26__11_43_53
✅ Category raw_data exported successfully with 2 sessions.

--- Aggregating for category: tidally_corrected ---
Processing session: ID_Jawa-18_datetime04-15-26__10_14_08
Processing session: ID_Jawa-17_datetime04-07-26__11_43_53
✅ Category tidally_corrected exported successfully with 2 sessions.
🏁 Ekspor Global selesai! Hasil ada di folder: global_portal_output
